# Check Code ZSCORE validation lab — V0.1

Independently recompute the fixed anthropometric cases from explicit LMS or percentile-spread parameters. This notebook does not import the TypeScript implementation. Passing is evidence for a governed candidate, not scientific approval or complete desktop parity.

In [ ]:
from pyodide.http import pyfetch
response = await pyfetch('../../validation-fixtures/check-code-zscore-v0.1.json')
response.raise_for_status()
fixture = await response.json()
fixture['operation'], fixture['referenceVersion']

In [ ]:
def lms(measurement, p):
    return ((measurement / p['M']) ** p['L'] - 1.0) / (p['L'] * p['S'])

def nchs_spread(measurement, p):
    lower = ((p['p50'] - p['p5']) / 1.65 + (p['p50'] - p['p10']) / 1.28 + (p['p50'] - p['p25']) / 0.67) / 3.0
    upper = ((p['p95'] - p['p50']) / 1.65 + (p['p90'] - p['p50']) / 1.28 + (p['p75'] - p['p50']) / 0.67) / 3.0
    spread = upper if measurement > p['p50'] else lower
    return (measurement - p['p50']) / spread

rows = []
for case in fixture['cases']:
    oracle = lms(case['measurement'], case['parameters']) if case['method'] == 'lms' else nchs_spread(case['measurement'], case['parameters'])
    assert abs(oracle - case['expected']) <= fixture['tolerance']
    rows.append({'reference': case['reference'], 'metric': case['metric'], 'expected': case['expected'], 'oracle': oracle})
rows

In [ ]:
assert {row['reference'] for row in rows} == {'CDC 2000', 'WHO 2006', 'WHO 2007', 'NCHS 1977'}
{'status': 'PASS', 'cases': len(rows), 'referenceVersion': fixture['referenceVersion'], 'scope': 'exact-row formula oracle'}